# Домашняя работа №3: Построение продвинутого пайплайна классификации для свёрточных сетей

В этой домашней работе мы продолжим работать с датасетом Tiny ImageNet и постараемся сильно улучшить предыдущий результат посредством построения более продвинутого пайплайна обучения без изменения самой модели. В рамках этого задания желательно продолжить с той же архитектурой, которую вы использовали в предыдущем домашнем задании, — так вы напрямую увидите, насколько сильное влияние оказывает сам тренировочный процесс и что не всегда стоит бежать менять архитектуру, столкнувшись с неудовлетворительным качеством работы сети :) Итак, приступим!

## Часть 0: Подготовка

Импортируем необходимые библиотеки.

In [1]:
!pip install fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 39.5 MB/s eta 0:00:00a 0:00:01


In [ ]:
import io
import os
from time import time
from typing import Union, Callable, Optional

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.v2 as transforms
import torchvision as tv
import pandas as pd
import numpy as np
import PIL.Image as Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary

from dataclasses import dataclass
import wandb
import tqdm


Скопируем пайплайн тренировки из предыдущего домашнего задания.

In [ ]:
# фиксируем сиды
def enable_determinism():
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True) # на этот раз зафиксируем алгоритмы, чтобы изменения точно не были случайными

def fix_seeds(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # torch.mps.manual_seed(seed)
    
def seed_worker(_):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
def myshow(img):
    # img = img * 0.3 + 0.3 # умножаем на std, прибавляем mean - в Normalize всё наоборот
    npimg = img.detach().numpy()
    fig = plt.figure(figsize=(16, 16))
    plt.imshow(npimg.transpose(1, 2, 0))

In [ ]:
def run_epoch(model: nn.Module, epoch, loader: DataLoader, criterion: Callable, optimizer: Optional[torch.optim.Optimizer] = None,\
              scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None, device: torch.device = torch.device("cpu")) -> torch.Tensor:
    all_labels, all_preds = [], []
    loss_epoch = 0.
    for batch in tqdm.tqdm(loader):
        images, labels = batch
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if scheduler is not None:
                scheduler.step()

        loss_epoch += loss.item()
        preds = torch.argmax(logits.softmax(dim=-1), dim=-1)
        if len(labels.size()) > 1:
            labels = labels.argmax(dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    loss_epoch /= len(loader)
    acc_epoch = (all_preds == all_labels).sum() / len(all_preds)

    if optimizer is not None:
        wandb.log({'lr': optimizer.param_groups[0]['lr']}, step=epoch)    
        wandb.log({'train loss': loss_epoch}, step=epoch)
        wandb.log({'train accuracy': acc_epoch * 100.0}, step=epoch)
    else:
        wandb.log({'test loss': loss_epoch}, step=epoch)
        wandb.log({'test accuracy': acc_epoch * 100.0}, step=epoch)

    return loss_epoch, acc_epoch

def train(model: nn.Module, n_epochs: int, train_loader: DataLoader, criterion: Callable, optimizer: torch.optim.Optimizer,
          scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None, val_loader: Optional[DataLoader] = None, val_freq: int = 10,\
          save_best: bool = True, save_name: str = 'model', device: torch.device = torch.device("cpu")) -> nn.Module:
    enable_validation = val_loader is not None
    best_val = 0.

    for epoch in range(n_epochs):
        timer_start = time()
        model.train()
        train_loss_epoch, train_acc_epoch = run_epoch(model, epoch, train_loader, criterion, optimizer, scheduler, device)

        print(f"Epoch {epoch+1}:")
        print(f"Train loss: {train_loss_epoch} | Train acc: {train_acc_epoch * 100}%")

        if enable_validation and epoch % val_freq == 0:
            model.eval()
            with torch.no_grad():
                val_loss_epoch, val_acc_epoch = run_epoch(model, epoch, val_loader, criterion, optimizer=None, scheduler=None, device=device)

            if save_best and val_acc_epoch >= best_val:
                best_val = val_acc_epoch
                model.to("cpu")
                torch.save(model.state_dict(), f"{save_name}.pth")
                model.to(device)

            print(f"Val loss: {val_loss_epoch} | Val acc: {val_acc_epoch * 100}%")

        print(f"Time spent on epoch: {time() - timer_start}")

    return model

Скопируйте класс датасета из предыдущего задания.

In [ ]:
class TinyImageNetDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: Callable):
        super().__init__()

        self._data = df
        self.transform = transform

    def __len__(self) -> int:
        return len(self._data)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        sample = self._data.iloc[idx]

        image = Image.open(io.BytesIO(sample['image.bytes']))
        if image.mode == 'L':
            image = image.convert('RGB')
        image = self.transform(image)

        label = torch.tensor(sample['label'], dtype=torch.long)
        return image, label


Скопируйте функцию `stratified_train_val_split` из предыдущего задания.

In [ ]:
def stratified_train_val_split(df: pd.DataFrame, train_share: float, seed: int = 42) -> tuple[pd.DataFrame, pd.DataFrame]:
    np.random.seed(seed)

    label_counts = df['label'].value_counts() # посчитаем число лейблов каждого класса
    train_counts = (label_counts * train_share).round().astype(int) # посчитаем, какая часть лейблов в каждом классе пойдёт на train

    train_indices, val_indices = [], []
    for label in label_counts.index:
        class_indices = df[df['label'] == label].index # получим индексы всех семплов данного класса
        
        shuffled_indices = np.random.permutation(class_indices) # перемешаем
        
        n_train = train_counts[label] # сколько должно быть семплов этого класса в трейне
        train_indices.extend(shuffled_indices[:n_train]) # первые n_train идут в train
        val_indices.extend(shuffled_indices[n_train:]) # остаток — в val
    
    train_df = df.loc[train_indices].copy()
    val_df = df.loc[val_indices].copy()
    
    train_df.sort_index(inplace=True)
    val_df.sort_index(inplace=True)

    return train_df, val_df

Скопируйте вашу архитектуру модели и необходимые блоки.

In [ ]:
class SqueezeExcitation(nn.Module):
    def __init__(self, in_channels: int, squeeze_rate: int) -> None:
        super().__init__()
        
        self.se_block = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, in_channels // squeeze_rate, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(in_channels // squeeze_rate, in_channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale = self.se_block(x)
        return scale * x
        

class MBConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, exp_channels: int, kernel_size: Union[int, tuple[int, int]],
                 padding: Union[int, tuple[int, int]], stride: Union[int, tuple[int, int]], non_linearity: str = 'RE',
                 se_block: bool = True, squeeze_rate: int = 16) -> None:
        super().__init__()

        self.non_linearity_bank = {'RE': nn.ReLU6, 'HS': nn.Hardswish}

        self.use_skip_connection = stride != 2 and in_channels == out_channels

        self.layers = []

        if exp_channels != in_channels:
            self.layers.append(nn.Conv2d(in_channels, exp_channels, kernel_size=1))
            self.layers.append(nn.BatchNorm2d(exp_channels))
            self.layers.append(self.non_linearity_bank[non_linearity]())
        
        self.layers.append(nn.Conv2d(exp_channels, exp_channels, kernel_size, stride, padding, groups=exp_channels))
        self.layers.append(nn.BatchNorm2d(exp_channels))
        self.layers.append(self.non_linearity_bank[non_linearity]())
        
        if se_block:
            self.layers.append(SqueezeExcitation(exp_channels, squeeze_rate))
        
        self.layers.append(nn.Conv2d(exp_channels, out_channels, kernel_size=1))
        self.layers.append(nn.BatchNorm2d(out_channels))
        
        self.layers = nn.Sequential(*self.layers)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.layers(x)
        # Если пространственный размер входного тензора не меняется, то прибавляем скип
        if self.use_skip_connection:
            out = out + x
        return out


class MobileNetV3(nn.Module):
    def __init__(self, in_channels: int, num_classes: int):
        super().__init__()

        self.in_channels = in_channels

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, 2, 1),
            nn.BatchNorm2d(16),
            nn.Hardswish()
        )

        self.feature_extractor = nn.Sequential(
            # bneck  in  out  exp k  p  s     NE    se
            MBConv2d(16, 16, 16, 3, 1, 1, 'RE', True),
            MBConv2d(16, 24, 64, 3, 1, 2, 'RE', False),
            MBConv2d(24, 24, 72, 3, 1, 1, 'HS', True),
            MBConv2d(24, 32, 72, 3, 1, 1, 'HS', True),
            MBConv2d(32, 64, 96, 3, 1, 2, 'HS', True),
            MBConv2d(64, 64, 128, 3, 1, 1, 'HS', True),
            MBConv2d(64, 128, 128, 3, 1, 1, 'HS', True),
            MBConv2d(128, 128, 256, 3, 1, 1, 'HS', True),
            MBConv2d(128, 128, 256, 3, 1, 1, 'HS', True),
            nn.Conv2d(128, 512, 1),
            nn.BatchNorm2d(512),
            nn.Hardswish(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512, 1024),
            nn.Hardswish(),
            nn.Linear(1024, num_classes),
        )

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.feature_extractor(x)
        x = torch.flatten(x, 1)
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.forward_features(x)
        x = self.classifier(x)
        return x

model = MobileNetV3(3, 200)
model_summary = summary(model, input_size=(1,3,64,64), verbose=True)

assert model_summary.total_params <= 1.5e6, "Слишком много параметров, уменьшите сеть"
assert model_summary.total_mult_adds <= 1e8, "Слишком высокая вычислительная сложность, оптимизируйте сеть"

Загрузим данные и разобьём их на train-/val-части.

In [ ]:
data_path = r"/kaggle/input/datasets/andreykurdyubov/tiny-imagenet/tiny_imagenet/train-00000-of-00001-1359597a978bc4fa.parquet" # замените на путь до .parquet файла с train частью датасета
df = pd.read_parquet(data_path, engine='fastparquet')
df.drop(columns=['image.path'], inplace=True)

train_df, val_df = stratified_train_val_split(df, train_share=0.9, seed=42)

In [ ]:
data_path_val = r"/kaggle/input/datasets/andreykurdyubov/tiny-imagenet/tiny_imagenet/valid-00000-of-00001-70d52db3c749a935.parquet" 
df_val = pd.read_parquet(data_path, engine='fastparquet')
df_val.drop(columns=['image.path'], inplace=True)

train_df_val, val_df_val = stratified_train_val_split(df_val, train_share=0.9, seed=42)

In [ ]:
df.size

## Часть 1: Пайплайн аугментации

Постройте пайплайн аугментации, примените подходы, разобранные на лекциях и практике, попробуйте свои идеи. Список аугментаций, доступный в torchvision, приведён тут: https://pytorch.org/vision/stable/transforms.html. В качестве одного из элементов пайплайна рекомендуем обратить внимание на RandAugment:

**RandAugment** — это алгоритм автоматического аугментирования изображений, разработанный Google Research. Его основная идея заключается в случайном применении набора простых операций преобразования изображений (таких, как поворот, изменение яркости, контраста, обрезка и т.д.). Оригинальная статья: https://arxiv.org/pdf/1909.13719v2.

Он довольно прост в настройке, т. к. принимает всего два гиперпараметра: *N* — количество аугментаций, применяемых за раз, *M* — сила каждой аугментации. Полный список аугментаций:
- RandomShear
- RandomTranslation
- RandomRotation
- RandomBrightness
- RandomColor
- RandomContrast
- Posterize
- Solarize
- Equalize
- AutoContrast

In [ ]:
train_transform = transforms.Compose([
    # transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True)
])

val_transform = transforms.Compose([
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True)
])

Рассмотрим также более продвинутые способы аугментации — **MixUp** и **CutMix**. Традиционные методы аугментации (например, поворот, масштабирование, изменение яркости) работают с одним изображением, применяя к нему различные преобразования. MixUp и CutMix же работают сразу с парами изображений и, как можно судить из названий, каким-то образом смешивают их.

Начнём с MixUp. Он действует следующим образом: берутся два изображения и их метки, затем создаётся новое изображение путём их линейного смешивания. Представьте, что у вас есть фотография кошки и фотография собаки. MixUp накладывает их друг на друга с определённым коэффициентом $\lambda$ (например, 0.6 кошка + 0.4 собака). При этом метка нового изображения также становится смешанной: [0.6, 0.4].

CutMix работает иначе: вместо смешивания всего изображения он вырезает прямоугольную область из одного изображения и вставляет её в другое. Метки также смешиваются, но пропорционально площади вырезанной области. Если мы вырезали 30% площади из изображения собаки и вставили в изображение кошки, метка будет [0.7, 0.3].

Дополнительное отличие этих двух методов от тех, которые мы использовали в первой части, — способ встраивания в пайплайн обучения. Традиционные аугментации обычно применяются на этапе предобработки данных перед началом формирования батча. MixUp и CutMix в свою очередь же работают с батчами, а не с индивидуальными картинками, поэтому требуют иной логики встраивания в пайплайн: самый простой способ — это применять эти аугментации непосредственно в тренировочном цикле, но в таком случае мы никак не пользуемся мультипроцессингом нашего дата-лоадера. Более оптимальный подход — передавать в data loader модифицированную collate_fn, где на батч применяются наши аугментации, поэтому напишем `collate_fn` для применения этих аугментаций. Документация по `collate_fn`: https://pytorch.org/docs/stable/data.html#working-with-collate-fn. По сути, когда включено автоматическое формирование батчей (дефолт), эта функция принимает на вход лист из семплов, полученных вызовом метода `__getitem__` у нашего датасета, а потом формирует из него батч. Стандартная `torch.utils.data.default_collate` просто стекает их по батч-дименшену и конвертит в `torch.Tensor`.

In [ ]:
# random.beta - распределение на отрезке [0, 1] 
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta

# Разные значения alpha
alphas = [0.1, 0.2, 0.5, 1.0, 2.0]
x = np.linspace(0, 1, 100)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for idx, alpha in enumerate(alphas):
    # Генерируем 10000 значений lam
    lam_samples = np.random.beta(alpha, alpha, 10000)
    
    # Показываем распределение
    axes[idx].hist(lam_samples, bins=50, density=True, alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'alpha = {alpha}')
    axes[idx].set_xlabel('lam (коэффициент смешивания)')
    axes[idx].set_ylabel('Плотность вероятности')
    axes[idx].set_xlim(0, 1)
    
    # Добавляем теоретическую кривую
    y = beta.pdf(x, alpha, alpha)
    axes[idx].plot(x, y, 'r-', linewidth=2)
    
    # Статистика
    axes[idx].text(0.05, 0.95, f'mean = {lam_samples.mean():.3f}\nstd = {lam_samples.std():.3f}', 
                   transform=axes[idx].transAxes, verticalalignment='top')

plt.tight_layout()
plt.show()

In [ ]:
# определите здесь CutMix/MixUp

def MixUp(batch, alpha=0.2):
    """
    Берем по 2 случайные картинки внутри батча и смешиваем их в некоторой пропорции
    """
    images = torch.stack([item[0] for item in batch])
    labels = torch.tensor([item[1] for item in batch])

    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = images.size(0)
    index = torch.randperm(batch_size)

    images_shuffled = images[index]
    labels_shuffled = labels[index]

    mixed_images = lam * images + (1 - lam) * images_shuffled
    return mixed_images, labels, labels_shuffled, lam


def MixCut(batch, alpha=0.2):
    """
    Делаем вставку одной картинки в другой
    """
    images = torch.stack([item[0] for item in batch])
    labels = torch.tensor([item[1] for item in batch])
    
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    
    batch_size = images.size(0)
    index = torch.randperm(batch_size)
    
    images_shuffled = images[index]
    labels_shuffled = labels[index]
    
    batch_size, channels, height, width = images.shape

    # Стандартная формула для CutMix
    cut_ratio = np.sqrt(1 - lam)  
    cut_h = int(height * cut_ratio)
    cut_w = int(width * cut_ratio)
    
    # Генерируем координаты
    cx = np.random.randint(0, width)
    cy = np.random.randint(0, height)
    
    # Вычисляем границы
    x1 = np.clip(cx - cut_w // 2, 0, width)
    x2 = np.clip(cx + cut_w // 2, 0, width)
    y1 = np.clip(cy - cut_h // 2, 0, height)
    y2 = np.clip(cy + cut_h // 2, 0, height)
    
    # Создаем маску
    mask = torch.ones((height, width), dtype=torch.float32)
    mask[y1:y2, x1:x2] = 0
    mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
    
    # Смешиваем изображения
    mixed_images = images * mask + images_shuffled * (1 - mask)
    
    # Пересчитываем lambda
    lam_adjusted = 1 - ((x2 - x1) * (y2 - y1)) / (width * height)
    
    return mixed_images, labels, labels_shuffled, lam_adjusted


def collate_fn(batch, alpha=0.2, choice_mixup=True):
    """
    True - MixUp
    False - MixCut
    None - 50/50
    """
    if choice_mixup is None:
        choice_mixup = np.random.random() > 0.5

    # 50/50
    if choice_mixup:
        return MixUp(batch, alpha)
    else:
        return MixCut(batch, alpha)

In [ ]:
# Проверка для MixCut
lam = 0.4
height = width = 8
size_h, size_w = np.ceil(lam * height).astype(int), np.ceil(lam * width).astype(int)
print(size_h, size_w)

start_h = np.random.randint(0, height - size_h)
start_w = np.random.randint(0, width - size_w)

mask_a = np.ones((height, width), dtype=int)
mask_a[start_h:start_h+size_h, start_w:start_w+size_w] = 0
mask_b = 1 - mask_a
mask_a, mask_b

In [ ]:
# конфиг с основными гиперпараметрами
@dataclass
class Config:
    seed: int = 24
    batch_size: int = 32
    img_size: int = 64
    n_epochs: int = 10
    lr: float = 1e-4

# не забывайте, что фиксировать заново сиды и создавать даталоадеры с ними нужно каждый раз, если хотите воспроизводимости
config = Config()
enable_determinism()
fix_seeds(config.seed)

generator = torch.Generator()
generator.manual_seed(config.seed)

train_transform = transforms.Compose([
    transforms.RandomCrop(size=(56, 56)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True),
    transforms.RandomErasing()
])

train_dataset = TinyImageNetDataset(train_df, train_transform)

# по датасетам создаем даталоадеры
trainloader = DataLoader(
    train_dataset, 
    batch_size=config.batch_size, 
    num_workers=4, 
    shuffle=True, 
    pin_memory=True,
    drop_last=True,
    worker_init_fn=seed_worker,
    collate_fn=lambda batch: collate_fn(batch, alpha=1, choice_mixup=True),
    generator=generator,
)

# заведем итератор по нашему датасету и возьмем из него случайный батч
trainiter = iter(trainloader)
images, labels_a, labels_b, lam = next(trainiter)

# отрисуем случайный батч
myshow(tv.utils.make_grid(images))

Отлично, теперь попробуйте построить собственный пайплайн аугментаций.

Примечание: аугментации выше даны для примера, необязательно использовать их, возможно, вы соберёте оптимальный набор из совершенно других методов.

Примечание 2: если будете использовать CutMix/MixUp, то рекомендуем увеличить количество эпох, поскольку они дают достаточно сильную регуляризацию.

In [ ]:
def mixup_cutmix_criterion(criterion, pred, labels_a, labels_b, lam):
    """
    Вычисление loss для MixUp/CutMix
    """
    return lam * criterion(pred, labels_a) + (1 - lam) * criterion(pred, labels_b)


def compute_metrics_for_cutmix(logits, labels_a, labels_b, lam):
    """
    Вычисление метрик для CutMix (более точный способ)
    """
    # Получаем вероятности
    probs = torch.softmax(logits, dim=-1)
    preds = torch.argmax(probs, dim=-1)
    
    correct_a = (preds == labels_a).float()
    simple_accuracy = correct_a.mean().item()
    
    return simple_accuracy, preds


def run_epoch_cutmix(model: nn.Module, epoch, loader: DataLoader, criterion: Callable, optimizer: Optional[torch.optim.Optimizer] = None,\
              scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None, device: torch.device = torch.device("cpu")) -> torch.Tensor:
    all_labels, all_preds = [], []
    loss_epoch = 0.
    acc_epoch = 0.
    
    for batch in tqdm.tqdm(loader):
        if optimizer is not None:
            images, labels_a, labels_b, lam = batch
            images, labels_a, labels_b = images.to(device), labels_a.to(device), labels_b.to(device)
    
            logits = model(images)
            loss = mixup_cutmix_criterion(criterion, logits, labels_a, labels_b, lam)
            preds = torch.argmax(torch.softmax(logits, dim=-1), dim=-1)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
            if scheduler is not None:
                scheduler.step()
    
            loss_epoch += loss.item()
            # simple_accuracy, preds = compute_metrics_for_cutmix(logits, labels_a, labels_b, lam)
    
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_a.cpu().numpy())
            
        else:
            images, labels = batch
            images, labels = images.to(device), labels.to(device)
    
            logits = model(images)
            loss = criterion(logits, labels)
            preds = torch.argmax(torch.softmax(logits, dim=-1), dim=-1)
  
            loss_epoch += loss.item()
      
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    loss_epoch /= len(loader)
    acc_epoch = (all_preds == all_labels).sum() / len(all_preds)

    if optimizer is not None:
        wandb.log({'lr': optimizer.param_groups[0]['lr']}, step=epoch)    
        wandb.log({'train loss': loss_epoch}, step=epoch)
        wandb.log({'train accuracy': acc_epoch * 100.0}, step=epoch)
    else:
        wandb.log({'test loss': loss_epoch}, step=epoch)
        wandb.log({'test accuracy': acc_epoch * 100.0}, step=epoch)

    return loss_epoch, acc_epoch

def train_cutmix(model: nn.Module, n_epochs: int, train_loader: DataLoader, criterion: Callable, optimizer: torch.optim.Optimizer,
          scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None, val_loader: Optional[DataLoader] = None, val_freq: int = 10,\
          save_best: bool = True, save_name: str = 'model', device: torch.device = torch.device("cpu")) -> nn.Module:
    enable_validation = val_loader is not None
    best_val = 0.

    for epoch in range(n_epochs):
        timer_start = time()
        model.train()
        train_loss_epoch, train_acc_epoch = run_epoch_cutmix(model, epoch, train_loader, criterion, optimizer, scheduler, device)

        print(f"Epoch {epoch+1}:")
        print(f"Train loss: {train_loss_epoch} | Train acc: {train_acc_epoch * 100}%")

        if enable_validation and epoch % val_freq == 0:
            model.eval()
            with torch.no_grad():
                val_loss_epoch, val_acc_epoch = run_epoch_cutmix(model, epoch, val_loader, criterion, optimizer=None, scheduler=None, device=device)

            if save_best and val_acc_epoch >= best_val:
                best_val = val_acc_epoch
                model.to("cpu")
                torch.save(model.state_dict(), f"{save_name}.pth")
                model.to(device)

            print(f"Val loss: {val_loss_epoch} | Val acc: {val_acc_epoch * 100}%")

        print(f"Time spent on epoch: {time() - timer_start}")

    return model

In [ ]:
logits = torch.rand((4, 8))
labels = torch.tensor([3, 6, 4, 2], dtype=int)
# labels = torch.zeros((4, 8))
# labels[0, 2] = labels[1, 1] = labels[2, 5] = labels[3, 7] = 1 
print(logits)
print(labels)
probs = torch.softmax(logits, dim=-1)
preds = torch.argmax(probs, dim=-1)
print(preds)
correct_a = (preds == labels)#.float()
print(correct_a)
correct_a = (preds == labels).float()
print(correct_a)
correct_a.mean()

In [ ]:
###########################################################################################

In [ ]:
# конфиг с основными гиперпараметрами
@dataclass
class Config:
    seed: int = 24
    batch_size: int = 100
    img_size: int = 64
    n_epochs: int = 20
    lr: float = 3e-4

# не забывайте, что фиксировать заново сиды и создавать даталоадеры с ними нужно каждый раз, если хотите воспроизводимости
config = Config()
enable_determinism()
fix_seeds(config.seed)

generator = torch.Generator()
generator.manual_seed(config.seed)

# transforms
train_transform = transforms.Compose([
    # transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.TrivialAugmentWide(),
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True)
])

val_transform = transforms.Compose([
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True)
])

train_dataset = TinyImageNetDataset(train_df, train_transform)
val_dataset = TinyImageNetDataset(val_df, val_transform)

# Если используете MixUp/CutMix, не забудьте добавить в train_loader collate_fn=collate_fn
train_loader = DataLoader(
    train_dataset, 
    batch_size=config.batch_size, 
    num_workers=4, 
    shuffle=True, 
    pin_memory=False,
    drop_last=True,
    worker_init_fn=seed_worker,
    # collate_fn=lambda batch: collate_fn(batch, alpha=0.3, choice_mixup=False),
    generator=generator,
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=config.batch_size, 
    pin_memory=False, 
    shuffle=False)
                           

n_epochs = config.n_epochs
criterion = nn.CrossEntropyLoss()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MobileNetV3(3, 200)
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.n_epochs * len(train_loader))

wandb.init(
    project="CV-hw3-TinyImageNet", 
    name="TrivialAug lr=3e-4 CosineScheduler", 
    config=config.__dict__
)

# p3 - triv aug 20 epochs
# p4 - MixUp alpha=0.3
# p5 - CutMix alpha=0.3
# p6 - Triv + cosine

model = train(model, n_epochs, train_loader, criterion, optimizer, scheduler=scheduler, val_loader=val_loader, val_freq=1, save_best=True, save_name="model_p6_cosine", device=device)
# model = train_cutmix(model, n_epochs, train_loader, criterion, optimizer, scheduler=None, val_loader=val_loader, val_freq=1, save_best=True, save_name="model_p5_CutMix", device=device)


wandb.finish()

In [ ]:
wandb.init(
    project="CV-hw3-TinyImageNet", 
    name="TrivialAug Cosine from p3", 
    config=config.__dict__
)

# p3 - triv aug 20 epochs
# p4 - MixUp alpha=0.3
# p5 - CutMix alpha=0.3
# p6 - Triv + cosine
# p7 - Triv + cosine from p3
# p8 - Triv + cosine from p7

state_dict = torch.load('/kaggle/working/model_p7_cosine.pth')
model.load_state_dict(state_dict)
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.n_epochs * len(train_loader))

model = train(model, n_epochs, train_loader, criterion, optimizer, scheduler=scheduler, val_loader=val_loader, val_freq=1, save_best=True, save_name="model_p8_cosine", device=device)
# model = train_cutmix(model, n_epochs, train_loader, criterion, optimizer, scheduler=None, val_loader=val_loader, val_freq=1, save_best=True, save_name="model_p5_CutMix", device=device)


wandb.finish()

In [ ]:
# Transforms автора
train_transform = transforms.Compose([
    transforms.RandomCrop(size=(56, 56)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True),
    transforms.RandomErasing()
])

Отлично! Можем увидеть, что даже добавление простых аугментаций значительно снижает переобучение модели и повышает итоговый скор. Загрузите код модели и веса в LMS для проверки в приватном тесте.

## Часть 2: Настройка процесса оптимизации модели

Теперь тренировочный датасет стал намного более вариативным, настало время перейти к настройке процесса оптимизации модели. В этой части ДЗ предлагаем вам подобрать подходящий оптимизатор, scheduler (если понадобится) и Learning Rate.

Скопируйте пайплайны аугментации, которые вы получили в прошлой части задания.

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(size=(56, 56)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True),
    transforms.RandomErasing()
])

val_transform = transforms.Compose([
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True)
])

In [ ]:
df_cc = pd.concat([df, df_val], ignore_index=True)
df_cc.size

In [17]:
import io
import os
from time import time
from typing import Union, Callable, Optional

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.v2 as transforms
import torchvision as tv
import pandas as pd
import numpy as np
import PIL.Image as Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary

from dataclasses import dataclass
import wandb
import tqdm

# фиксируем сиды
def enable_determinism():
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True) # на этот раз зафиксируем алгоритмы, чтобы изменения точно не были случайными

def fix_seeds(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # torch.mps.manual_seed(seed)
    
def seed_worker(_):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def stratified_train_val_split(df: pd.DataFrame, train_share: float, seed: int = 42) -> tuple[pd.DataFrame, pd.DataFrame]:
    np.random.seed(seed)

    label_counts = df['label'].value_counts() # посчитаем число лейблов каждого класса
    train_counts = (label_counts * train_share).round().astype(int) # посчитаем, какая часть лейблов в каждом классе пойдёт на train

    train_indices, val_indices = [], []
    for label in label_counts.index:
        class_indices = df[df['label'] == label].index # получим индексы всех семплов данного класса
        
        shuffled_indices = np.random.permutation(class_indices) # перемешаем
        
        n_train = train_counts[label] # сколько должно быть семплов этого класса в трейне
        train_indices.extend(shuffled_indices[:n_train]) # первые n_train идут в train
        val_indices.extend(shuffled_indices[n_train:]) # остаток — в val
    
    train_df = df.loc[train_indices].copy()
    val_df = df.loc[val_indices].copy()
    
    train_df.sort_index(inplace=True)
    val_df.sort_index(inplace=True)

    return train_df, val_df


class TinyImageNetDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: Callable):
        super().__init__()

        self._data = df
        self.transform = transform

    def __len__(self) -> int:
        return len(self._data)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        sample = self._data.iloc[idx]

        image = Image.open(io.BytesIO(sample['image.bytes']))
        if image.mode == 'L':
            image = image.convert('RGB')
        image = self.transform(image)

        label = torch.tensor(sample['label'], dtype=torch.long)
        return image, label

class SqueezeExcitation(nn.Module):
    def __init__(self, in_channels: int, squeeze_rate: int) -> None:
        super().__init__()
        
        self.se_block = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, in_channels // squeeze_rate, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(in_channels // squeeze_rate, in_channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale = self.se_block(x)
        return scale * x
        

class MBConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, exp_channels: int, kernel_size: Union[int, tuple[int, int]],
                 padding: Union[int, tuple[int, int]], stride: Union[int, tuple[int, int]], non_linearity: str = 'RE',
                 se_block: bool = True, squeeze_rate: int = 16) -> None:
        super().__init__()

        self.non_linearity_bank = {'RE': nn.ReLU6, 'HS': nn.Hardswish}

        self.use_skip_connection = stride != 2 and in_channels == out_channels

        self.layers = []

        if exp_channels != in_channels:
            self.layers.append(nn.Conv2d(in_channels, exp_channels, kernel_size=1))
            self.layers.append(nn.BatchNorm2d(exp_channels))
            self.layers.append(self.non_linearity_bank[non_linearity]())
        
        self.layers.append(nn.Conv2d(exp_channels, exp_channels, kernel_size, stride, padding, groups=exp_channels))
        self.layers.append(nn.BatchNorm2d(exp_channels))
        self.layers.append(self.non_linearity_bank[non_linearity]())
        
        if se_block:
            self.layers.append(SqueezeExcitation(exp_channels, squeeze_rate))
        
        self.layers.append(nn.Conv2d(exp_channels, out_channels, kernel_size=1))
        self.layers.append(nn.BatchNorm2d(out_channels))
        
        self.layers = nn.Sequential(*self.layers)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.layers(x)
        # Если пространственный размер входного тензора не меняется, то прибавляем скип
        if self.use_skip_connection:
            out = out + x
        return out


class MobileNetV3(nn.Module):
    def __init__(self, in_channels: int, num_classes: int):
        super().__init__()

        self.in_channels = in_channels

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, 2, 1),
            nn.BatchNorm2d(16),
            nn.Hardswish()
        )

        self.feature_extractor = nn.Sequential(
            # bneck  in  out  exp k  p  s     NE    se
            MBConv2d(16, 16, 16, 3, 1, 1, 'RE', True),
            MBConv2d(16, 24, 64, 3, 1, 2, 'RE', False),
            MBConv2d(24, 24, 72, 3, 1, 1, 'HS', True),
            MBConv2d(24, 32, 72, 3, 1, 1, 'HS', True),
            MBConv2d(32, 64, 96, 3, 1, 2, 'HS', True),
            MBConv2d(64, 64, 128, 3, 1, 1, 'HS', True),
            MBConv2d(64, 128, 128, 3, 1, 1, 'HS', True),
            MBConv2d(128, 128, 256, 3, 1, 1, 'HS', True),
            MBConv2d(128, 128, 256, 3, 1, 1, 'HS', True),
            nn.Conv2d(128, 512, 1),
            nn.BatchNorm2d(512),
            nn.Hardswish(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512, 1024),
            nn.Hardswish(),
            nn.Linear(1024, num_classes),
        )

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.feature_extractor(x)
        x = torch.flatten(x, 1)
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.forward_features(x)
        x = self.classifier(x)
        return x


# определите здесь CutMix/MixUp

def MixUp(batch, alpha=0.2):
    """
    Берем по 2 случайные картинки внутри батча и смешиваем их в некоторой пропорции
    """
    images = torch.stack([item[0] for item in batch])
    labels = torch.tensor([item[1] for item in batch])

    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = images.size(0)
    index = torch.randperm(batch_size)

    images_shuffled = images[index]
    labels_shuffled = labels[index]

    mixed_images = lam * images + (1 - lam) * images_shuffled
    return mixed_images, labels, labels_shuffled, lam


def MixCut(batch, alpha=0.2):
    """
    Делаем вставку одной картинки в другой
    """
    images = torch.stack([item[0] for item in batch])
    labels = torch.tensor([item[1] for item in batch])
    
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    
    batch_size = images.size(0)
    index = torch.randperm(batch_size)
    
    images_shuffled = images[index]
    labels_shuffled = labels[index]
    
    batch_size, channels, height, width = images.shape

    # Стандартная формула для CutMix
    cut_ratio = np.sqrt(1 - lam)  
    cut_h = int(height * cut_ratio)
    cut_w = int(width * cut_ratio)
    
    # Генерируем координаты
    cx = np.random.randint(0, width)
    cy = np.random.randint(0, height)
    
    # Вычисляем границы
    x1 = np.clip(cx - cut_w // 2, 0, width)
    x2 = np.clip(cx + cut_w // 2, 0, width)
    y1 = np.clip(cy - cut_h // 2, 0, height)
    y2 = np.clip(cy + cut_h // 2, 0, height)
    
    # Создаем маску
    mask = torch.ones((height, width), dtype=torch.float32)
    mask[y1:y2, x1:x2] = 0
    mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
    
    # Смешиваем изображения
    mixed_images = images * mask + images_shuffled * (1 - mask)
    
    # Пересчитываем lambda
    lam_adjusted = 1 - ((x2 - x1) * (y2 - y1)) / (width * height)
    
    return mixed_images, labels, labels_shuffled, lam_adjusted


def collate_fn(batch, alpha=0.2, choice_mixup=True):
    """
    True - MixUp
    False - MixCut
    None - 50/50
    """
    if choice_mixup is None:
        choice_mixup = np.random.random() > 0.5

    # 50/50
    if choice_mixup:
        return MixUp(batch, alpha)
    else:
        return MixCut(batch, alpha)


def mixup_cutmix_criterion(criterion, pred, labels_a, labels_b, lam):
    """
    Вычисление loss для MixUp/CutMix
    """
    return lam * criterion(pred, labels_a) + (1 - lam) * criterion(pred, labels_b)


def run_epoch_cutmix(model: nn.Module, epoch, loader: DataLoader, criterion: Callable, optimizer: Optional[torch.optim.Optimizer] = None,\
              scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None, device: torch.device = torch.device("cpu")) -> torch.Tensor:
    all_labels, all_preds = [], []
    loss_epoch = 0.
    acc_epoch = 0.
    
    for batch in tqdm.tqdm(loader):
        if isinstance(batch, tuple) and len(batch) == 4:
            images, labels_a, labels_b, lam = batch
            images, labels_a, labels_b = images.to(device), labels_a.to(device), labels_b.to(device)

            logits = model(images)
            loss = mixup_cutmix_criterion(criterion, logits, labels_a, labels_b, lam)
                
            preds = torch.argmax(torch.softmax(logits, dim=-1), dim=-1)
            all_labels.extend(labels_a.cpu().numpy())
        else:
            images, labels = batch
            images, labels = images.to(device), labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)
                
            preds = torch.argmax(torch.softmax(logits, dim=-1), dim=-1)
            all_labels.extend(labels.cpu().numpy())
             
        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            if scheduler is not None:
                scheduler.step()

        loss_epoch += loss.item()
        all_preds.extend(preds.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    loss_epoch /= len(loader)
    acc_epoch = (all_preds == all_labels).sum() / len(all_preds)

    if optimizer is not None:
        wandb.log({'lr': optimizer.param_groups[0]['lr']}, step=epoch)    
        wandb.log({'train loss': loss_epoch}, step=epoch)
        wandb.log({'train accuracy': acc_epoch * 100.0}, step=epoch)
    else:
        wandb.log({'test loss': loss_epoch}, step=epoch)
        wandb.log({'test accuracy': acc_epoch * 100.0}, step=epoch)

    return loss_epoch, acc_epoch


def train_cutmix(model: nn.Module, n_epochs: int, train_loader: DataLoader, criterion: Callable, optimizer: torch.optim.Optimizer,
          scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None, val_loader: Optional[DataLoader] = None, val_freq: int = 10,\
          save_best: bool = True, save_name: str = 'model', device: torch.device = torch.device("cpu")) -> nn.Module:
    enable_validation = val_loader is not None
    best_val = 0.

    for epoch in range(n_epochs):
        timer_start = time()
        model.train()
        train_loss_epoch, train_acc_epoch = run_epoch_cutmix(model, epoch, train_loader, criterion, optimizer, scheduler, device)

        print(f"Epoch {epoch+1}:")
        print(f"Train loss: {train_loss_epoch} | Train acc: {train_acc_epoch * 100}%")

        if enable_validation and epoch % val_freq == 0:
            model.eval()
            with torch.no_grad():
                val_loss_epoch, val_acc_epoch = run_epoch_cutmix(model, epoch, val_loader, criterion, optimizer=None, scheduler=None, device=device)

            if save_best and val_acc_epoch >= best_val:
                best_val = val_acc_epoch
                model.to("cpu")
                torch.save(model.state_dict(), f"{save_name}.pth")
                model.to(device)

            print(f"Val loss: {val_loss_epoch} | Val acc: {val_acc_epoch * 100}%")

        print(f"Time spent on epoch: {time() - timer_start}")

    return model


class FocalLoss(nn.Module):
    def __init__(self, alpha: Union[float, list, tuple] = 1., gamma: float = 2., reduction: str = 'mean'):
        super().__init__()

        assert reduction in ['none', 'mean', 'sum'], f"{reduction} should be one of {['none', 'mean', 'sum']}"

        self.alpha = torch.tensor(alpha) if isinstance(alpha, (list, tuple)) else alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_probs = torch.log_softmax(inputs, -1)
        probs = torch.exp(log_probs)

        log_probs = torch.gather(log_probs, -1, targets.unsqueeze(1))
        probs = torch.gather(probs, -1, targets.unsqueeze(1))

        focal_loss = -self.alpha*(1-probs)**self.gamma*log_probs
        focal_loss = focal_loss.sum(dim=-1)

        if self.reduction == 'none':
            return focal_loss
        elif self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()


class GeneralizedCrossEntropy(nn.Module):
    def __init__(self, q: float = 0.7, reduction: str = 'mean'):
        super().__init__()

        assert q <= 1.0 and q > 0., "Incorrect q value"
        assert reduction in ['none', 'mean', 'sum'], f"{reduction} should be one of {['none', 'mean', 'sum']}"

        self.q = q
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.softmax(inputs, dim=-1)
        probs = torch.gather(probs, -1, targets.unsqueeze(1))

        loss = (1 - probs**self.q)/self.q

        if self.reduction == 'none':
            return loss
        elif self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()


# конфиг с основными гиперпараметрами
@dataclass
class Config:
    seed: int = 24
    batch_size: int = 100
    img_size: int = 64
    n_epochs: int = 200
    lr: float = 1e-1

# fix seeds
config = Config()
enable_determinism()
fix_seeds(config.seed)

generator = torch.Generator()
generator.manual_seed(config.seed)

# author transforms
train_transform = transforms.Compose([
    transforms.RandomCrop(size=(56, 56)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandAugment(num_ops=2, magnitude=11),
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True),
    transforms.RandomErasing()
])

val_transform = transforms.Compose([
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True)
])


# load val data
data_path_val = r"/kaggle/input/datasets/andreykurdyubov/tiny-imagenet/tiny_imagenet/valid-00000-of-00001-70d52db3c749a935.parquet" 
df_val = pd.read_parquet(data_path_val, engine='fastparquet')
df_val.drop(columns=['image.path'], inplace=True)

# load data
data_path = r"/kaggle/input/datasets/andreykurdyubov/tiny-imagenet/tiny_imagenet/train-00000-of-00001-1359597a978bc4fa.parquet"
df = pd.read_parquet(data_path, engine='fastparquet')
df.drop(columns=['image.path'], inplace=True)

df = pd.concat([df, df_val], ignore_index=True)
train_df, val_df = stratified_train_val_split(df, train_share=0.95, seed=42)

train_dataset = TinyImageNetDataset(train_df, train_transform)
val_dataset = TinyImageNetDataset(val_df, val_transform)

# Если используете MixUp/CutMix, не забудьте добавить в train_loader collate_fn=collate_fn
train_loader = DataLoader(
    train_dataset, 
    batch_size=config.batch_size, 
    num_workers=4, 
    shuffle=True, 
    pin_memory=True,
    drop_last=True,
    worker_init_fn=seed_worker,
    # collate_fn=lambda batch: collate_fn(batch, alpha=0.3, choice_mixup=None),
    generator=generator,
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=config.batch_size, 
    pin_memory=True, 
    shuffle=False)
                           

n_epochs = config.n_epochs
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
# criterion = FocalLoss(alpha=1, gamma=1)
# criterion = GeneralizedCrossEntropy()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MobileNetV3(3, 200)
model.to(device)

wandb.init(
    project="CV-hw3-TinyImageNet", 
    name="Author Augs alldata SGD lr=0.1", 
    config=config.__dict__
)

# p3 - triv aug 20 epochs
# p4 - MixUp alpha=0.3
# p5 - CutMix alpha=0.3
# p6 - Triv + cosine
# p7 - Triv + cosine from p3
# p8 - Triv + cosine from p7
# p9 - author augs from p8
# p11 label_smoothing =0.05
# p12 df_val
# p13 df_val+df_train
# p15 all data 200 epochs
# p16 focal loss alph=4 gamma=2
# p17 focal loss alph=1 gamma=2
# p18 focal loss alph=1 gamma=1
# p19 GCE q=0.7 lr=2e-4
# p20 GCE q=0.7 lr=7e-4
# p21 SGD CE lr=0.1


# state_dict = torch.load('/kaggle/working/model_p13_author_aug_bs100.pth')
# model.load_state_dict(state_dict)

# optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
optimizer = torch.optim.SGD(model.parameters(), lr=config.lr, momentum=0.9)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=config.n_epochs * len(train_loader),
    # eta_min=1e-5
)

model = train_cutmix(model, n_epochs, train_loader, criterion, optimizer, scheduler=scheduler, val_loader=val_loader, val_freq=1, save_best=True, save_name="model_p21", device=device)

wandb.finish()

lr,█████▇▇▇▇▆▆▆▅▅▄▄▃▃▂▂▁
test accuracy,▁▂▃▃▄▄▄▅▅▆▆▆▆▇▇▇█████
test loss,█▇▆▆▅▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁
train accuracy,▁▂▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇███
train loss,█▇▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▁▁▁
lr,0.00068
test accuracy,26.28571
test loss,1.02524
train accuracy,22.33621
train loss,1.08935


100%|██████████| 1044/1044 [00:55<00:00, 18.95it/s]


Epoch 1:
Train loss: 4.892597071512449 | Train acc: 3.684865900383142%


100%|██████████| 56/56 [00:03<00:00, 17.11it/s]


Val loss: 4.586669389690671 | Val acc: 6.892857142857142%
Time spent on epoch: 58.4452166557312


...

Epoch 50:
Train loss: 2.0972107158995223 | Train acc: 56.76149425287357%


100%|██████████| 56/56 [00:03<00:00, 17.34it/s]


Val loss: 2.40819322850023 | Val acc: 51.73214285714286%
Time spent on epoch: 58.654311656951904


 15%|█▌        | 157/1044 [00:08<00:48, 18.18it/s]


KeyboardInterrupt: 

### 2026.05.14 Попробуем запустить Distibuted Data Parallel

In [ ]:
%%writefile train_ddp.py
   
import io
import os
from time import time
from time import sleep
from typing import Union, Callable, Optional

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.v2 as transforms
import torchvision as tv
import pandas as pd
import numpy as np
import PIL.Image as Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary

from dataclasses import dataclass
import wandb
import tqdm

import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

# # preferences
# def enable_determinism():
#     os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
#     torch.use_deterministic_algorithms(True) # на этот раз зафиксируем алгоритмы, чтобы изменения точно не были случайными

def fix_seeds(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # torch.mps.manual_seed(seed)
    
def seed_worker(_):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def setup_ddp(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'  # любой свободный порт
    dist.init_process_group(backend='nccl', rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup_ddp():
    dist.destroy_process_group()

# dataset
class TinyImageNetDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: Callable):
        super().__init__()

        self._data = df
        self.transform = transform

    def __len__(self) -> int:
        return len(self._data)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        sample = self._data.iloc[idx]

        image = Image.open(io.BytesIO(sample['image.bytes']))
        if image.mode == 'L':
            image = image.convert('RGB')
        image = self.transform(image)

        label = torch.tensor(sample['label'], dtype=torch.long)
        return image, label

# train-val split
def stratified_train_val_split(df: pd.DataFrame, train_share: float, seed: int = 42) -> tuple[pd.DataFrame, pd.DataFrame]:
    np.random.seed(seed)

    label_counts = df['label'].value_counts() # посчитаем число лейблов каждого класса
    train_counts = (label_counts * train_share).round().astype(int) # посчитаем, какая часть лейблов в каждом классе пойдёт на train

    train_indices, val_indices = [], []
    for label in label_counts.index:
        class_indices = df[df['label'] == label].index # получим индексы всех семплов данного класса
        
        shuffled_indices = np.random.permutation(class_indices) # перемешаем
        
        n_train = train_counts[label] # сколько должно быть семплов этого класса в трейне
        train_indices.extend(shuffled_indices[:n_train]) # первые n_train идут в train
        val_indices.extend(shuffled_indices[n_train:]) # остаток — в val
    
    train_df = df.loc[train_indices].copy()
    val_df = df.loc[val_indices].copy()
    
    train_df.sort_index(inplace=True)
    val_df.sort_index(inplace=True)

    return train_df, val_df

# net blocks
class SqueezeExcitation(nn.Module):
    def __init__(self, in_channels: int, squeeze_rate: int) -> None:
        super().__init__()
        
        self.se_block = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, in_channels // squeeze_rate, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(in_channels // squeeze_rate, in_channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale = self.se_block(x)
        return scale * x
        

class MBConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, exp_channels: int, kernel_size: Union[int, tuple[int, int]],
                 padding: Union[int, tuple[int, int]], stride: Union[int, tuple[int, int]], non_linearity: str = 'RE',
                 se_block: bool = True, squeeze_rate: int = 16) -> None:
        super().__init__()

        self.non_linearity_bank = {'RE': nn.ReLU6, 'HS': nn.Hardswish}

        self.use_skip_connection = stride != 2 and in_channels == out_channels

        self.layers = []

        if exp_channels != in_channels:
            self.layers.append(nn.Conv2d(in_channels, exp_channels, kernel_size=1))
            self.layers.append(nn.BatchNorm2d(exp_channels))
            self.layers.append(self.non_linearity_bank[non_linearity]())
        
        self.layers.append(nn.Conv2d(exp_channels, exp_channels, kernel_size, stride, padding, groups=exp_channels))
        self.layers.append(nn.BatchNorm2d(exp_channels))
        self.layers.append(self.non_linearity_bank[non_linearity]())
        
        if se_block:
            self.layers.append(SqueezeExcitation(exp_channels, squeeze_rate))
        
        self.layers.append(nn.Conv2d(exp_channels, out_channels, kernel_size=1))
        self.layers.append(nn.BatchNorm2d(out_channels))
        
        self.layers = nn.Sequential(*self.layers)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.layers(x)
        # Если пространственный размер входного тензора не меняется, то прибавляем скип
        if self.use_skip_connection:
            out = out + x
        return out

# main model
class MobileNetV3(nn.Module):
    def __init__(self, in_channels: int, num_classes: int):
        super().__init__()

        self.in_channels = in_channels

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, 2, 1),
            nn.BatchNorm2d(16),
            nn.Hardswish()
        )

        self.feature_extractor = nn.Sequential(
            # bneck  in  out  exp k  p  s     NE    se
            MBConv2d(16, 16, 16, 3, 1, 1, 'RE', True),
            MBConv2d(16, 24, 64, 3, 1, 2, 'RE', False),
            MBConv2d(24, 24, 72, 3, 1, 1, 'HS', True),
            MBConv2d(24, 32, 72, 3, 1, 1, 'HS', True),
            MBConv2d(32, 64, 96, 3, 1, 2, 'HS', True),
            MBConv2d(64, 64, 128, 3, 1, 1, 'HS', True),
            MBConv2d(64, 128, 128, 3, 1, 1, 'HS', True),
            MBConv2d(128, 128, 256, 3, 1, 1, 'HS', True),
            MBConv2d(128, 128, 256, 3, 1, 1, 'HS', True),
            nn.Conv2d(128, 512, 1),
            nn.BatchNorm2d(512),
            nn.Hardswish(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512, 1024),
            nn.Hardswish(),
            nn.Linear(1024, num_classes),
        )

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.feature_extractor(x)
        x = torch.flatten(x, 1)
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.forward_features(x)
        x = self.classifier(x)
        return x

# mixing augmentations
def MixUp(batch, alpha=0.2):
    """
    Берем по 2 случайные картинки внутри батча и смешиваем их в некоторой пропорции
    """
    images = torch.stack([item[0] for item in batch])
    labels = torch.tensor([item[1] for item in batch])

    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = images.size(0)
    index = torch.randperm(batch_size)

    images_shuffled = images[index]
    labels_shuffled = labels[index]

    mixed_images = lam * images + (1 - lam) * images_shuffled
    return mixed_images, labels, labels_shuffled, lam


def MixCut(batch, alpha=0.2):
    """
    Делаем вставку одной картинки в другой
    """
    images = torch.stack([item[0] for item in batch])
    labels = torch.tensor([item[1] for item in batch])
    
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    
    batch_size = images.size(0)
    index = torch.randperm(batch_size)
    
    images_shuffled = images[index]
    labels_shuffled = labels[index]
    
    batch_size, channels, height, width = images.shape

    # Стандартная формула для CutMix
    cut_ratio = np.sqrt(1 - lam)  
    cut_h = int(height * cut_ratio)
    cut_w = int(width * cut_ratio)
    
    # Генерируем координаты
    cx = np.random.randint(0, width)
    cy = np.random.randint(0, height)
    
    # Вычисляем границы
    x1 = np.clip(cx - cut_w // 2, 0, width)
    x2 = np.clip(cx + cut_w // 2, 0, width)
    y1 = np.clip(cy - cut_h // 2, 0, height)
    y2 = np.clip(cy + cut_h // 2, 0, height)
    
    # Создаем маску
    mask = torch.ones((height, width), dtype=torch.float32)
    mask[y1:y2, x1:x2] = 0
    mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
    
    # Смешиваем изображения
    mixed_images = images * mask + images_shuffled * (1 - mask)
    
    # Пересчитываем lambda
    lam_adjusted = 1 - ((x2 - x1) * (y2 - y1)) / (width * height)
    
    return mixed_images, labels, labels_shuffled, lam_adjusted

def collate_fn(batch, alpha=0.2, choice_mixup=True):
    """
    True - MixUp
    False - MixCut
    None - 50/50
    """
    if choice_mixup is None:
        choice_mixup = np.random.random() > 0.5

    # 50/50
    if choice_mixup:
        return MixUp(batch, alpha)
    else:
        return MixCut(batch, alpha)


def mixup_cutmix_criterion(criterion, pred, labels_a, labels_b, lam):
    """
    Вычисление loss для MixUp/CutMix
    """
    return lam * criterion(pred, labels_a) + (1 - lam) * criterion(pred, labels_b)


# training usinng Distributed Data Parallel
def run_epoch_ddp(model, epoch, loader, criterion, optimizer=None, scheduler=None, 
                  device=torch.device("cpu"), rank=0, scaler=None):
    all_labels, all_preds = [], []
    loss_epoch = 0.
    
    for batch in tqdm.tqdm(loader, disable=rank != 0):
        # Обработка mixup/cutmix или обычных данных
        if isinstance(batch, tuple) and len(batch) == 4:
            images, labels_a, labels_b, lam = batch
            images, labels_a, labels_b = images.to(device), labels_a.to(device), labels_b.to(device)

            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                    loss = mixup_cutmix_criterion(criterion, logits, labels_a, labels_b, lam)
            else:
                logits = model(images)
                loss = mixup_cutmix_criterion(criterion, logits, labels_a, labels_b, lam)
                
            preds = torch.argmax(torch.softmax(logits, dim=-1), dim=-1)
            all_labels.extend(labels_a.cpu().numpy())
        else:
            images, labels = batch
            images, labels = images.to(device), labels.to(device)

            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                    loss = criterion(logits, labels)
            else:
                logits = model(images)
                loss = criterion(logits, labels)
                
            preds = torch.argmax(torch.softmax(logits, dim=-1), dim=-1)
            all_labels.extend(labels.cpu().numpy())

        if optimizer is not None:
            optimizer.zero_grad()

            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            
            if scheduler is not None:
                scheduler.step()

        loss_epoch += loss.item()
        all_preds.extend(preds.cpu().numpy())

    loss_epoch /= len(loader)
    all_preds, all_labels = np.array(all_preds), np.array(all_labels)
    acc_epoch = (all_preds == all_labels).sum() / len(all_preds)

    if rank == 0:
        if optimizer is not None:
            wandb.log({'lr': optimizer.param_groups[0]['lr']}, step=epoch)    
            wandb.log({'train loss': loss_epoch}, step=epoch)
            wandb.log({'train accuracy': acc_epoch * 100.0}, step=epoch)
        else:
            wandb.log({'test loss': loss_epoch}, step=epoch)
            wandb.log({'test accuracy': acc_epoch * 100.0}, step=epoch)
    return loss_epoch, acc_epoch

def train_ddp(model, n_epochs, train_loader, criterion, optimizer, scheduler=None, 
              val_loader=None, val_freq=10, save_best=True, save_name='model', 
              device=torch.device("cpu"), rank=0, scaler=None):
    enable_validation = val_loader is not None
    best_val = 0.
    
    for epoch in range(n_epochs):
        timer_start = time()
        model.train()
        train_sampler = train_loader.sampler
        if isinstance(train_sampler, DistributedSampler):
            train_sampler.set_epoch(epoch)  # 🔑 Важно для DDP: перемешивание данных между эпохами
            
        train_loss, train_acc = run_epoch_ddp(model, epoch, train_loader, criterion, optimizer, scheduler, device, rank, scaler=scaler)

        if rank == 0:
            print(f"Epoch {epoch+1}: Train loss: {train_loss:.4f} | Train acc: {train_acc*100:.2f}%")

        if enable_validation and epoch % val_freq == 0:
            model.eval()
            with torch.no_grad():
                val_loss, val_acc = run_epoch_ddp(model, epoch, val_loader, criterion, 
                                                  optimizer=None, scheduler=None, device=device, rank=rank, scaler=scaler)
            if rank == 0:
                print(f"Val loss: {val_loss:.4f} | Val acc: {val_acc*100:.2f}%")
                
            if rank == 0 and save_best and val_acc >= best_val:
                best_val = val_acc
                # Сохраняем state_dict без DDP обёртки
                torch.save(model.module.state_dict(), f"{save_name}.pth")
                print(f"💾 Saved best model at epoch {epoch+1}")

        if rank == 0:
            print(f"⏱ Time spent on epoch: {time() - timer_start:.2f}s")
            
    return model

# config
@dataclass
class Config:
    seed: int = 24
    batch_size: int = 400
    img_size: int = 64
    n_epochs: int = 20
    lr: float = 1e-3

def main():
    # 🔑 torchrun автоматически выставляет эти переменные
    rank = int(os.environ['RANK'])
    world_size = int(os.environ['WORLD_SIZE'])
    local_rank = int(os.environ['LOCAL_RANK'])
    
    dist.init_process_group(backend='nccl', rank=rank, world_size=world_size, device_id=local_rank)
    torch.cuda.set_device(local_rank)
    device = torch.device(f'cuda:{local_rank}')
    
    config = Config()
    # ✅ Заменяем enable_determinism() на безопасный вариант для DDP
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    random.seed(config.seed + rank)
    np.random.seed(config.seed + rank)
    torch.manual_seed(config.seed + rank)
    torch.cuda.manual_seed(config.seed + rank)
    
    # --- Загрузка данных (в каждом процессе) ---
    data_path = "/kaggle/input/datasets/andreykurdyubov/tiny-imagenet/tiny_imagenet/train-00000-of-00001-1359597a978bc4fa.parquet"     
    df = pd.read_parquet(data_path, engine='fastparquet')
    df.drop(columns=['image.path'], inplace=True)
    train_df, val_df = stratified_train_val_split(df, train_share=0.9, seed=config.seed)
    
    # transforms
    train_transform = transforms.Compose([
        transforms.RandomCrop(size=(56, 56)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.PILToTensor(),
        transforms.ToDtype(dtype=torch.float32, scale=True),
        transforms.RandomErasing()
    ])

    val_transform = transforms.Compose([
        transforms.PILToTensor(),
        transforms.ToDtype(dtype=torch.float32, scale=True)
    ])
    
    train_ds = TinyImageNetDataset(train_df, train_transform)
    val_ds = TinyImageNetDataset(val_df, val_transform)
    
    # 🔑 Samplers вместо shuffle=True
    train_sampler = DistributedSampler(
        train_ds,
        num_replicas=world_size, 
        rank=rank, 
        shuffle=True, 
        seed=config.seed,
     )
    
    val_sampler = DistributedSampler(
        val_ds, 
        num_replicas=world_size, 
        rank=rank, 
        shuffle=False, 
        seed=config.seed
    )
    
    # 🔑 shuffle=False, т.к. порядок задаёт Sampler
    train_loader = DataLoader(
        train_ds, 
        batch_size=config.batch_size, 
        sampler=train_sampler,
        num_workers=4, 
        pin_memory=False,
        persistent_workers=True,
        drop_last=True, 
        worker_init_fn=seed_worker,
        # collate_fn=lambda batch: collate_fn(batch, alpha=0.3, choice_mixup=True),
    )
    
    val_loader = DataLoader(
        val_ds, 
        batch_size=config.batch_size, 
        sampler=val_sampler,                           
        num_workers=4, 
        pin_memory=True
    )
    
    model = MobileNetV3(3, 200).to(device)
    model = DDP(model, device_ids=[local_rank])
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    # scaler = torch.amp.GradScaler('cuda')
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=config.n_epochs * len(train_loader),
    eta_min=1e-5)
    
    if rank == 0:
        wandb.init(project="CV-hw3-TinyImageNet", name="DDP Author Augs bs=400 x2", config=config.__dict__)
        
    model = train_ddp(model, config.n_epochs, train_loader, criterion, optimizer, scheduler,
              val_loader=val_loader, val_freq=1, save_best=True, save_name="model_ddp", 
              device=device, rank=rank, scaler=None)
    
    if rank == 0:
        wandb.finish()
    dist.destroy_process_group()

if __name__ == '__main__':
    main()

In [ ]:
!pip install fastparquet
import wandb
wandb.login()

In [ ]:
!torchrun --nproc_per_node=2 train_ddp.py

Отправьте код модели и натренированный вес в LMS.

## Основной результат: DDP не дал большого прироста (а в определенных сценариях даже проиграл) в скорости обучения. Причина - CPU bottleneck. Подготовка данных и перекидыване на GPU занимает слишком много времени. Варьирование размера батча на каждом из 2-х GPU от 25 до 1000 не помогло - начиная с 50 общее время примерно постоянно.Трюки с кэшированием данных в RAM и применение AMP.autocast scaler не помогает в данном случае - поскольоку дело не в скорости вычислений, а в подготовке данных.

## Часть 3: Focal Loss

В следующих 3 частях начнём адресовать проблему шумных лейблов через лосс-функции. В этой части предлагаем реализовать Focal Loss, про который уже рассказывали на лекции. Обычно он используется для датасетов с сильным дисбалансом классов, но его также можно применять и для кейса с шумными лейблами, поскольку он снижает их влияние на тренировочный процесс, если выставить более низкое значение гиперпараметра $\gamma$. Напомним формулу лосса:
$$ \text{FL}(p_{t,c}) = -\sum_{c=1}^C \alpha_c(1-p_{t,c})^\gamma y_c\log(p_{t,c}). $$

In [5]:
from typing import Union, Callable, Optional

class FocalLoss(nn.Module):
    def __init__(self, alpha: Union[float, list, tuple] = 1., gamma: float = 2., reduction: str = 'mean'):
        super().__init__()

        assert reduction in ['none', 'mean', 'sum'], f"{reduction} should be one of {['none', 'mean', 'sum']}"

        self.alpha = torch.tensor(alpha) if isinstance(alpha, (list, tuple)) else alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_probs = torch.log_softmax(inputs, -1)
        probs = torch.exp(log_probs)

        log_probs = torch.gather(log_probs, -1, targets.unsqueeze(1))
        probs = torch.gather(probs, -1, targets.unsqueeze(1))

        focal_loss = -self.alpha*(1-probs)**self.gamma*log_probs
        focal_loss = focal_loss.sum(dim=-1)

        if self.reduction == 'none':
            return focal_loss
        elif self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()

In [6]:
test_logits = torch.tensor([[1.27, 0.15, 0.03], [0.37, 0.15, 0.33], [0.16, 0.09, 1.73]])
test_labels = torch.tensor([0, 2, 2])

loss = FocalLoss()
assert torch.isclose(loss(test_logits, test_labels), torch.tensor(0.1823), rtol=1e-3), "Incorrect value"

In [4]:
import torch
import numpy as np
import torch.nn as nn

CE = nn.CrossEntropyLoss()
logits = torch.rand((4, 10))
labels = torch.randint(10, size=(4,), dtype=torch.long)
print(logits)
print(labels)
print("========gather=======")
print(torch.gather(logits, 1, labels.unsqueeze(1)))
print("========gather=======")
print(CE(logits, labels))

def softmax(inp):
    out = torch.exp(inp)
    out = out / out.sum(dim=1, keepdim=True)
    return out

sm = softmax(logits)
print("==================softmax===========")
print(sm)
print(torch.softmax(logits, dim=1))
print(sm.sum(dim=1))
print("==================softmax===========")

ohlab = torch.zeros(size=logits.shape)
for k in range(ohlab.shape[0]):
    ohlab[k][labels[k]] = 1
print(ohlab)

loss = - ohlab * torch.log(softmax(logits))
loss = loss.sum(dim=1).mean()
print(loss)
# for i in range(len(labels)):
#     loss += -np.log(logits)

tensor([[0.3036, 0.8265, 0.4000, 0.7368, 0.6551, 0.6620, 0.0458, 0.3381, 0.8900,
         0.9143],
        [0.9780, 0.0608, 0.3183, 0.4010, 0.7239, 0.9919, 0.4891, 0.7436, 0.0718,
         0.6117],
        [0.8136, 0.3008, 0.0327, 0.1603, 0.9260, 0.4639, 0.6111, 0.1200, 0.9765,
         0.4794],
        [0.3945, 0.9316, 0.5300, 0.8443, 0.4793, 0.7857, 0.4006, 0.9788, 0.3487,
         0.2182]])
tensor([7, 5, 8, 7])
========gather=======
tensor([[0.3381],
        [0.9919],
        [0.9765],
        [0.9788]])
========gather=======
tensor(2.0727)
==================softmax===========
tensor([[0.0734, 0.1238, 0.0808, 0.1131, 0.1043, 0.1050, 0.0567, 0.0759, 0.1319,
         0.1351],
        [0.1477, 0.0590, 0.0764, 0.0830, 0.1146, 0.1498, 0.0906, 0.1169, 0.0597,
         0.1024],
        [0.1314, 0.0787, 0.0602, 0.0684, 0.1470, 0.0926, 0.1073, 0.0657, 0.1547,
         0.0941],
        [0.0795, 0.1360, 0.0910, 0.1246, 0.0865, 0.1175, 0.0799, 0.1425, 0.0759,
         0.0666]])
tensor([[0.0734,

In [ ]:
Отлично! Отправьте код в LMS на проверку.

## Часть 4: Generalized Cross Entropy Loss

Функция GCE была предложена как обобщение стандартной кросс-энтропии, которое делает модель более устойчивой к шуму в данных и неправильным меткам. Формула:
$$ L_{GCE}(p, y) = \frac{(1 - p_y^q)}{q}, $$
где $p_y$ — предсказанная вероятность принадлежности к классу $y$, $q$ — гиперпараметр, который варьируется от 0 до 1. Некоторые свойства этой лосс-функции:
1. Когда $q \to 1$, функция приближается к MAE, т. е. $\lim_{q \to 1} L_{GCE} = 1 - p_y$.
2. Когда $q \to 0$, функция приближается к обычной CE, т. е. $\lim_{q \to 0} L_{GCE} = -\log(p_y)$ (правило Лопиталя).

Идея в том, что, контролируя гиперпараметр $q$, мы можем изменять чувствительность лосса к шуму, т. к. MAE более устойчива к выбросам. Ссылка на оригинальную статью: https://arxiv.org/abs/1805.07836. Предлагаем имплементировать GCE ниже.

In [13]:
class GeneralizedCrossEntropy(nn.Module):
    def __init__(self, q: float = 0.7, reduction: str = 'mean'):
        super().__init__()

        assert q <= 1.0 and q > 0., "Incorrect q value"
        assert reduction in ['none', 'mean', 'sum'], f"{reduction} should be one of {['none', 'mean', 'sum']}"

        self.q = q
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.softmax(inputs, dim=-1)
        probs = torch.gather(probs, -1, targets.unsqueeze(1))

        loss = (1 - probs**self.q)/self.q

        if self.reduction == 'none':
            return loss
        elif self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()

In [14]:
test_logits = torch.tensor([[1.27, 0.15, 0.03], [0.37, 0.15, 0.33], [0.16, 0.09, 1.73]])
test_labels = torch.tensor([0, 2, 2])

loss = GeneralizedCrossEntropy(q = 0.7)
assert torch.isclose(loss(test_logits, test_labels), torch.tensor(0.4850), rtol=1e-3), "Incorrect value"

Загрузите код в LMS на проверку.

## Часть 5: Подбор лосс-функции

Предлагаем вам опробовать реализованные выше лосс-функции, а также Label Smoothing в деле. Скопируйте пайплайн обучения из части 2 и подберите лучшую лосс-функцию.

Примечание: оптимальная настройка лосс-функции — весьма сложная задача, поэтому в данной части от вас ожидается, скорее, валидация работоспособности реализованных вами лосс-функций из частей 3 и 4, а также проверка подхода с Label Smoothing в деле. Если не получится добиться улучшения, то решение с обычной CE должно тоже подойти.

In [ ]:
train_transform = transforms.Compose([
    ...
])

val_transform = transforms.Compose([
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True)
])

train_dataset, val_dataset = TinyImageNetDataset(train_df, train_transform), TinyImageNetDataset(val_df, val_transform)

train_loader, val_loader = DataLoader(train_dataset, batch_size=200, shuffle=True, pin_memory=True, drop_last=False),\
                           DataLoader(val_dataset, batch_size=200, pin_memory=True, shuffle=False)

n_epochs = 30

criterion = ...

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MobileNetV3(3, 200)
model.to(device)

optimizer = ...
lr_scheduler = ...

model = train(model, n_epochs, train_loader, criterion, optimizer, lr_scheduler, val_loader, val_freq=1, save_name='model_p3_ls', device=device)

Загрузите код модели и её веса в LMS.

## Часть 6: Ансамблирование модели

Рассмотрим ещё одну технику по улучшению качества модели — построение ансамбля модели из её весов в предыдущих эпохах. Два основных способа: Stochastic Weight Averaging (SWA) и Exponential Moving Average (EMA).

### Exponential Moving Average

Метод фактически предлагает вместо одного набора весов n-й эпохи брать их сглаженные значения, которые также учитывают наборы весов предыдущих эпох, т. е.
$$ \theta_{EMA} = \beta \theta_{EMA_{prev}} + (1-\beta) \theta_{current}, $$
где $\theta_{EMA}$ — сглаженный набор весов, $\theta_{current}$ — набор весов в текущей эпохе, $\beta$ — гиперпараметр, отвечающий за «силу» сглаживания.

### Stochastic Weight Averaging

Основная идея SWA связана с ландшафтом функции потерь нейронной сети. Представьте, что функция потерь — это горный рельеф, где мы ищем самую глубокую долину (глобальный минимум). Традиционное обучение с помощью стохастического градиентного спуска (SGD) похоже на спуск с горы в тумане: мы делаем шаги в направлении спуска, но можем застрять в локальном минимуме. SWA предлагает другой подход: вместо того чтобы использовать веса модели из последней эпохи обучения, мы собираем веса из разных точек траектории обучения и усредняем их. Пошагово алгоритм выполняет следующие действия:
1. Сначала модель обучается обычным способом (например, с помощью SGD) до момента, когда функция потерь начинает колебаться вокруг некоторого значения. Это означает, что мы достигли области с хорошими решениями.
2. После этого мы можем использовать циклический или постоянный Learning Rate. При циклическом подходе LR периодически меняется между заданными значениями, что позволяет модели исследовать разные области пространства решений.
3. На этом этапе мы периодически сохраняем веса модели. В конце обучения все собранные веса усредняются.

Более подробное объяснение можно найти, например, в блоге PyTorch — https://pytorch.org/blog/pytorch-1.6-now-includes-stochastic-weight-averaging/ — или в оригинальной статье — https://arxiv.org/abs/1803.05407. Примечание: метод можно использовать не только с SGD, но и с тем же Adam.

Документация PyTorch по работе с этими методами: https://pytorch.org/docs/stable/optim.html#weight-averaging-swa-and-ema. Предлагаем вам попробовать их в деле.

Внесите нужные изменения в функции для обучения модели, ориентируйтесь на документацию PyTorch:

In [ ]:
def run_epoch(model: nn.Module, loader: DataLoader, criterion: Callable, optimizer: Optional[torch.optim.Optimizer] = None,\
              scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None, ens_method: Optional[str] = None,\
              ens_model: Optional[torch.optim.swa_utils.AveragedModel] = None, ens_start: bool = False, device: torch.device = torch.device("cpu")) -> torch.Tensor:
    all_labels, all_preds = [], []
    loss_epoch = 0.
    for batch in loader:
        images, labels = batch
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            ...

            if scheduler is not None and (ens_method != 'swa' or not ens_start):
                scheduler.step()

        loss_epoch += loss.item()
        preds = torch.argmax(logits.softmax(dim=-1), dim=-1)

        all_preds = np.concatenate((all_preds, preds.cpu().numpy()))
        all_labels = np.concatenate((all_labels, labels.cpu().numpy()))

    loss_epoch /= len(loader)
    acc_epoch = (all_preds == all_labels).sum() / len(all_preds)

    return loss_epoch, acc_epoch

def train(model: nn.Module, n_epochs: int, train_loader: DataLoader, criterion: Callable, optimizer: torch.optim.Optimizer,
          scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None, val_loader: Optional[DataLoader] = None, val_freq: int = 10, ens_method: Optional[str] = None,\
          ens_model: Optional[torch.optim.swa_utils.AveragedModel] = None, ens_scheduler: Optional[torch.optim.swa_utils.SWALR] = None, ens_start_epoch: Optional[int] = None,\
          save_best: bool = True, save_name: str = 'model', device: torch.device = torch.device("cpu")) -> nn.Module:
    enable_validation = val_loader is not None
    best_val = 0.

    for epoch in range(n_epochs):
        timer_start = time()

        model.train()

        ens_start = (epoch > ens_start_epoch) if ens_start_epoch is not None else False
        train_loss_epoch, train_acc_epoch = run_epoch(model, train_loader, criterion, optimizer, scheduler, device)

        ...

        print(f"Epoch {epoch+1}:")
        print(f"Train loss: {train_loss_epoch} | Train acc: {train_acc_epoch * 100}%")

        if enable_validation and epoch % val_freq == 0:
            model.eval()
            with torch.no_grad():
                val_loss_epoch, val_acc_epoch = run_epoch(model, val_loader, criterion, optimizer=None, scheduler=None, device=device)

            if save_best and val_acc_epoch >= best_val:
                best_val = val_acc_epoch
                model.to("cpu")
                torch.save(model.state_dict(), f"{save_name}.pth")
                model.to(device)

            print(f"Val loss: {val_loss_epoch} | Val acc: {val_acc_epoch * 100}%")

        print(f"Time spent on epoch: {time() - timer_start}")

    return model

Возьмите пайплайн обучения из части 5 и добавьте EMA/SWA. Примечание: `ens_method` будет либо 'ema', либо 'swa'.

In [ ]:
train_transform = transforms.Compose([
    ...
])

val_transform = transforms.Compose([
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True)
])

train_dataset, val_dataset = TinyImageNetDataset(train_df, train_transform), TinyImageNetDataset(val_df, val_transform)

train_loader, val_loader = DataLoader(train_dataset, batch_size=200, shuffle=True, pin_memory=True, drop_last=False),\
                           DataLoader(val_dataset, batch_size=200, pin_memory=True, shuffle=False)

n_epochs = ...

criterion = ...

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MobileNetV3(3, 200)
model.to(device)

optimizer = ...
lr_scheduler = ...

...

model = train(model, n_epochs, train_loader, criterion, optimizer, lr_scheduler, val_loader, val_freq=1, save_name='model_p4', device=device)

...

Загрузите код модели и веса Averaged модели в LMS. Веса сохраняйте так же, как и с обычной моделью: `torch.save(ens_model.state_dict(), ...)`.

## Заключение

Итак, в этом домашнем задании мы рассмотрели основные подходы к построению продвинутого пайплайна для задачи классификации. В случае успешного выполнения всех заданий вы обнаружите, что только за счёт тюнинга процесса обучения можно улучшить метрику качества почти в 2 раза. Стоит отметить, что в реальных задачах вы, скорее всего, будете строить именно такие «продвинутые» пайплайны, где каждую компоненту тренировочного процесса нужно будет подтюнить для достижения наилучшего результата. За рамками данного ДЗ остались подходы вроде transfer learning, дистилляции и т. д., часть из них будет разобрана позднее, а пока можете почитать про эти техники самостоятельно.

In [ ]:
# Настройки обучения автора
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5, foreach=False, fused=True)
lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=2e-3, div_factor=10, total_steps=n_epochs*len(train_loader))